# Kaggle Jupyter Notebook

This notebook fetches the data for the challenge https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data.

It downloads the label/metadata CSVs, the **58 fully-labeled studies** (out of 4,407) and a small **sample of DICOM slices** for them - the full dataset is 569 GB.

Everything lands in the repo-root `data/` folder (never in `notebooks/`), which is git-ignored.

The Kaggle download endpoint is heavily rate-limited (HTTP 429). Every download retries with backoff, and the notebook is safe to re-run: anything already on disk is skipped.


### SETUP 

**in your .env file:**
--------------------

In [ ]:
# KAGGLE_USERNAME= #YOUR USERNAME HERE
# KAGGLE_API_TOKEN= #YOUR TOKEN HERE


### PYTHON INSTALL
--------------------
Install the Kaggle API, dotenv for environment variables, and pandas<br>


In [ ]:
!pip install -q kaggle python-dotenv pandas


### AUTHENTIFICATION:

In [ ]:
import os
import time
import zipfile
import shutil
import tempfile
import json
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv


# --- Repo root -------------------------------------------------------------
# This notebook lives in <repo>/notebooks, but the data must always land in
# <repo>/data. Never rely on the cwd, or you end up with notebooks/data.
def find_repo_root(start=None):
    p = (start or Path.cwd()).resolve()
    for candidate in (p, *p.parents):
        if (candidate / ".git").exists() or (candidate / "requirements.txt").exists():
            return candidate
    return p


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# 1. Load variables from the repo-root .env
load_dotenv(REPO_ROOT / ".env")

# 2. Get variables using the EXACT uppercase names from your .env
kaggle_user = os.environ.get("KAGGLE_USERNAME")
kaggle_key = os.environ.get("KAGGLE_API_TOKEN")

# 3. Check if they were found before setting them
if not kaggle_user or not kaggle_key:
    raise ValueError(
        "Credentials not found! Check that " + str(REPO_ROOT / ".env") + " exists and "
        "defines KAGGLE_USERNAME and KAGGLE_API_TOKEN."
    )

# 4. Map to exactly what the Kaggle API expects under the hood
os.environ["KAGGLE_USERNAME"] = kaggle_user
os.environ["KAGGLE_KEY"] = kaggle_key

# 5. Import and authenticate
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

print("Kaggle Authentication Successful!")
print("Repo root :", REPO_ROOT)
print("Data dir  :", DATA_DIR)

# Warn about a stray data folder left over from running this from notebooks/
stray = REPO_ROOT / "notebooks" / "data"
if stray.exists():
    print("\nWARNING: stray", stray, "exists - the real data dir is", DATA_DIR)


### Data setup and directory.

In [ ]:
COMP_NAME = "rsna-knee-abnormality-detection"

# Small metadata/label files - always fetched in full.
CSV_FILES = [
    "train.csv",
    "train_series.csv",
    "test.csv",
    "test_series.csv",
    "sample_submission.csv",
]

# The competition ships 4,407 studies / 569 GB, of which 58 are fully labeled.
# We take those 58 studies and only a handful of slices each, as a sample.
N_LABELED_STUDIES = 58
MAX_SERIES_PER_STUDY = 3   # the X / Y / Z views
SLICES_PER_SERIES = 5      # slices sampled per series


def list_competition_files(max_pages=2, page_size=200, prefixes=None, page_token=None):
    """Page through the competition file list.

    Returns (names, next_page_token, pages_read). There are millions of .dcm
    files, so always cap max_pages; `prefixes` keeps only paths we care about
    so we do not hold the whole listing in memory.
    """
    names, pages = [], 0
    for _ in range(max_pages):
        resp = api.competition_list_files(COMP_NAME, page_token=page_token, page_size=page_size)
        batch = getattr(resp, "files", resp) or []
        for f in batch:
            name = getattr(f, "name", str(f))
            if prefixes is None or name.startswith(prefixes):
                names.append(name)
        pages += 1
        page_token = getattr(resp, "next_page_token", None)
        if not page_token:
            break
    return names, page_token, pages


peek, more_token, _ = list_competition_files(max_pages=2)

counts = {}
for n in peek:
    key = n.split("/")[0]
    counts[key] = counts.get(key, 0) + 1

print("Peeked at", len(peek), "files" + (" (more remain)" if more_token else "") + ". Top-level entries:")
for k in sorted(counts):
    print(" -", k + ":", counts[k], "file(s) seen")
print("\nDICOM layout: <split>_series/<study_id>/<series_id>/<instance_id>.dcm")


### DOWNLOAD

Safe to interrupt and re-run: every download is retried with backoff on HTTP 429, and files already present in `data/` are skipped.


In [ ]:
# ---------------------------------------------------------------------------
# Rate-limit-safe download helper.
#
# The Kaggle competition download endpoint answers HTTP 429 (Too Many
# Requests) very aggressively, so every fetch retries with exponential
# backoff. Re-running this notebook is cheap: whatever is already on disk is
# skipped, so an interrupted run just continues where it stopped.
# ---------------------------------------------------------------------------
TMP_DIR = DATA_DIR / "_tmp"


def fetch(remote_path, target_path, retries=6, base_delay=10.0):
    """Download one competition file to target_path.

    Returns (status, path) with status one of "cached", "downloaded" or
    "failed: ...". The Kaggle API writes a flat basename (and sometimes a
    .zip), so the file is staged in a temp dir and then moved to the exact
    target path, which preserves the study/series folder structure.
    """
    target_path = Path(target_path)
    if target_path.exists() and target_path.stat().st_size > 0:
        return "cached", target_path

    target_path.parent.mkdir(parents=True, exist_ok=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)
    staging = Path(tempfile.mkdtemp(prefix="kg_", dir=TMP_DIR))
    try:
        last_err = None
        for attempt in range(retries):
            try:
                api.competition_download_file(COMP_NAME, remote_path, path=str(staging), quiet=True)
                last_err = None
                break
            except Exception as exc:          # 429s and 404s land here
                last_err = exc
                if attempt < retries - 1:
                    time.sleep(base_delay * (2 ** attempt))
        if last_err is not None:
            return "failed: " + str(last_err)[:70], None

        got = [p for p in staging.rglob("*") if p.is_file()]
        if len(got) == 1 and got[0].suffix == ".zip":
            with zipfile.ZipFile(got[0]) as zf:
                zf.extractall(staging)
            got[0].unlink()
            got = [p for p in staging.rglob("*") if p.is_file()]
        if len(got) != 1:
            return "failed: expected 1 file, got " + str(len(got)), None

        shutil.move(str(got[0]), str(target_path))
        return "downloaded", target_path
    finally:
        shutil.rmtree(staging, ignore_errors=True)


def fetch_all(jobs, label, every=1):
    """Run a list of (remote_path, target_path) jobs, printing % progress."""
    total = len(jobs)
    results = []
    for i, (remote, target) in enumerate(jobs, start=1):
        status, path = fetch(remote, target)
        results.append({"remote": remote, "status": status, "path": path})
        if i % every == 0 or i == total or not status.startswith(("cached", "downloaded")):
            pct = 100.0 * i / total if total else 100.0
            print("[{:>4}/{}] {:5.1f}%  {:<30} {}".format(i, total, pct, status, remote))
    ok = sum(1 for r in results if r["status"] in ("cached", "downloaded"))
    print("\n" + label + ": " + str(ok) + "/" + str(total) + " available locally.")
    if ok < total:
        print("Some files failed (usually HTTP 429). Wait a few minutes and re-run this cell -")
        print("already-downloaded files are skipped.")
    return results


print("Helpers ready. Downloads go to:", DATA_DIR)


#### Step 1 - label & metadata CSVs

In [ ]:
# --- Step 1: the label / metadata CSVs -------------------------------------
csv_results = fetch_all([(name, DATA_DIR / name) for name in CSV_FILES], "CSV files")


#### Step 2 - the 58 fully-labeled studies

In [ ]:
# --- Step 2: find the fully-labeled studies --------------------------------
# Most of the 4,407 rows in train.csv carry a report but no numeric labels;
# only 58 studies are fully labeled. Select them by "all label columns
# present", NOT by .head(58).
train_path = DATA_DIR / "train.csv"
if not train_path.exists():
    raise FileNotFoundError(
        str(train_path) + " is missing - re-run the CSV cell above (HTTP 429 is likely)."
    )

df_train = pd.read_csv(train_path)
print("train.csv:", df_train.shape[0], "rows x", df_train.shape[1], "columns")

study_col = next((c for c in df_train.columns if c.lower() in ("study_id", "studyinstanceuid", "study")), df_train.columns[0])
label_cols = [
    c
    for c in df_train.columns
    if c != study_col and pd.api.types.is_numeric_dtype(df_train[c])
]
print("study id column:", study_col)
print("numeric label columns:", len(label_cols))

if label_cols:
    filled = df_train[label_cols].notna().sum(axis=1)
    df_labeled = df_train[filled == filled.max()]
else:
    filled = None
    df_labeled = df_train.head(N_LABELED_STUDIES)

print("\nfully-labeled rows found:", len(df_labeled), "(expected", str(N_LABELED_STUDIES) + ")")
if len(df_labeled) != N_LABELED_STUDIES:
    print("Heads-up: count differs from the documented 58 - taking the first",
          N_LABELED_STUDIES, "best-labeled rows.")
    df_labeled = df_labeled.head(N_LABELED_STUDIES)

labeled_study_ids = df_labeled[study_col].astype(str).tolist()
print("\n--- labeled studies (head) ---")
display(df_labeled.head())


#### Step 3 - locate their DICOM slices

In [ ]:
# --- Step 3: resolve the .dcm paths for those studies ----------------------
# The file-list API has no prefix filter, so we page through it and keep only
# the paths belonging to our labeled studies. Progress is cached to disk, so a
# re-run resumes from the last page instead of starting over.
INDEX_PATH = DATA_DIR / "_index" / "train_series_index.json"
MAX_INDEX_PAGES = 200          # 200 pages x 200 files = 40k paths per run


def build_study_index(target_study_ids, max_pages=MAX_INDEX_PAGES, page_size=200):
    state = {"page_token": None, "pages_read": 0, "complete": False, "files": {}}
    if INDEX_PATH.exists():
        state.update(json.loads(INDEX_PATH.read_text()))

    targets = set(str(s) for s in target_study_ids)
    files = state["files"]

    if state["complete"]:
        print("Index already complete:", state["pages_read"], "pages read.")
        return files

    token = state["page_token"]
    for page in range(max_pages):
        names, token, _ = list_competition_files(
            max_pages=1, page_size=page_size, prefixes=("train_series/",), page_token=token
        )
        for name in names:
            parts = name.split("/")
            if len(parts) == 4 and parts[3].endswith(".dcm") and parts[1] in targets:
                files.setdefault(parts[1], {}).setdefault(parts[2], [])
                if name not in files[parts[1]][parts[2]]:
                    files[parts[1]][parts[2]].append(name)
        state["pages_read"] += 1
        if not token:
            state["complete"] = True
            break
        if (page + 1) % 20 == 0:
            print("  ...", state["pages_read"], "pages read,", len(files), "of",
                  len(targets), "target studies located")

    state["page_token"] = token
    INDEX_PATH.parent.mkdir(parents=True, exist_ok=True)
    INDEX_PATH.write_text(json.dumps(state))

    print("pages read this run:", state["pages_read"], "| listing exhausted:", state["complete"])
    print("target studies located:", len(files), "/", len(targets))
    if not state["complete"] and len(files) < len(targets):
        print("Listing not exhausted yet - re-run this cell to continue indexing"
              " (state cached in " + str(INDEX_PATH) + ").")
    return files


study_index = build_study_index(labeled_study_ids)


#### Step 4 - download the sampled slices

In [ ]:
# --- Step 4: download the sampled slices ----------------------------------
jobs = []
for study_id, series_map in study_index.items():
    for series_id in sorted(series_map)[:MAX_SERIES_PER_STUDY]:
        for remote in sorted(series_map[series_id])[:SLICES_PER_SERIES]:
            jobs.append((remote, DATA_DIR / remote))

print(len(jobs), "slice(s) queued from", len(study_index), "study/ies",
      "(" + str(MAX_SERIES_PER_STUDY), "series x", SLICES_PER_SERIES, "slices max each)\n")

dcm_results = fetch_all(jobs, "Sample DICOM slices", every=5) if jobs else []
if not jobs:
    print("Nothing queued - run the indexing cell above first.")


### RESULT - what landed on disk

In [ ]:
# --- Step 5: what is on disk ----------------------------------------------
print("Data directory:", DATA_DIR, "\n")

csvs = sorted(p for p in DATA_DIR.glob("*.csv"))
print("--- datasets downloaded ---")
for p in csvs:
    print("  {:<24} {:>10,} bytes".format(p.name, p.stat().st_size))

dcms = list(DATA_DIR.rglob("*.dcm"))
total_bytes = sum(p.stat().st_size for p in dcms)
studies = {p.relative_to(DATA_DIR).parts[1] for p in dcms if len(p.relative_to(DATA_DIR).parts) == 4}
print("\n  {} .dcm slice(s) across {} study/ies, {:,} bytes".format(len(dcms), len(studies), total_bytes))

# --- heads ---
for name in ("train.csv", "train_series.csv", "test.csv", "test_series.csv", "sample_submission.csv"):
    path = DATA_DIR / name
    if path.exists():
        df = pd.read_csv(path)
        print("\n--- " + name + " head (" + str(len(df)) + " rows, " + str(len(df.columns)) + " cols) ---")
        display(df.head())
    else:
        print("\n" + name + " not found.")
